# IRIS graph experiment: independent RGB decoding

Select a GPU runtime. This tests whether decoupling generation from decoding helps native serving on one GPU. CUDA correctness and performance are pending. Run qualification first and send its evidence before the audit or timing cells. See [Step 5d](https://github.com/Artifex2002/mstar/blob/codex/iris/docs/design/iris-step-05d.md). Use new output names on reruns; update the download cell too.


In [ ]:
%pip install -q uv


In [ ]:
%%bash
set -euo pipefail
if [ ! -e /content/mstar-iris ]; then
  git clone --branch codex/iris --single-branch https://github.com/Artifex2002/mstar.git /content/mstar-iris
fi
cd /content/mstar-iris
git switch codex/iris
git pull --ff-only origin codex/iris
if [ ! -e .venv ]; then
  uv venv --python 3.12 .venv
fi
export PATH="$HOME/.cargo/bin:$PWD/.venv/bin:$PATH"
if ! command -v rustup >/dev/null 2>&1; then
  curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal
fi
uv pip install --python .venv/bin/python --torch-backend=auto \
  -r scripts/iris/requirements-cuda-tests.txt -e '.[iris,metrics]' maturin
env -u CONDA_PREFIX VIRTUAL_ENV="$PWD/.venv" \
  .venv/bin/maturin develop --release --locked -m rust/Cargo.toml
.venv/bin/python -c 'import mstar_rust; print(mstar_rust.__version__)'

mkdir -p .ci-results/iris
if [ ! -e .ci-results/iris/reference ]; then
  git clone https://github.com/eloialonso/iris.git .ci-results/iris/reference
fi
git -C .ci-results/iris/reference checkout --detach 24326aaaa283c527f42b89b44cfdecf2665a7a16

if ! command -v ss >/dev/null 2>&1; then
  apt-get -qq update
  apt-get -qq install -y iproute2
fi


## 1. Correctness first

Both eager and captured pipeline servers must match the existing serial local baseline, finish concurrent requests and pass abort/reuse/cleanup checks. A failure blocks measurement. This starts two servers sequentially and can take several minutes. Send `graph-qualify-01` using the download cell even on failure.


In [ ]:
%%bash
set -euo pipefail
cd /content/mstar-iris
source .venv/bin/activate
mkdir -p .ci-results/iris
OMP_NUM_THREADS=2 python -m benchmark.iris.graph_ab --stage qualify \
  --download-checkpoint --output .ci-results/iris/graph-qualify-01 \
  2>&1 | tee .ci-results/iris/graph-qualify-01.log


## 2. Untimed scheduling audit — after correctness passes

This starts all four layouts/execution modes with worker phase instrumentation. These diagnostic timings are excluded from performance comparisons. Inspect `worker-phases.txt` and batching histograms before attributing a mechanism. Do not update the checkout between qualification and the later cells.


In [ ]:
%%bash
set -euo pipefail
cd /content/mstar-iris
source .venv/bin/activate
OMP_NUM_THREADS=2 python -m benchmark.iris.graph_ab --stage audit \
  --qualification .ci-results/iris/graph-qualify-01/qualification \
  --output .ci-results/iris/graph-audit-01 \
  2>&1 | tee .ci-results/iris/graph-audit-01.log


## 3. Controlled HTTP A/B — after reviewing the audit

Three rounds, four arms: eager serial/pipeline and captured serial/pipeline. Fresh server per arm, same weights/actions/H20/GPU/precision/sampler within capture mode, C=1/2, five warmup waves then ten requests per cell. This starts twelve servers; expect many minutes. Phase instrumentation is disabled. The unchanged official standalone arm is not rerun.


In [ ]:
%%bash
set -euo pipefail
cd /content/mstar-iris
source .venv/bin/activate
OMP_NUM_THREADS=2 python -m benchmark.iris.graph_ab --stage measure \
  --qualification .ci-results/iris/graph-qualify-01/qualification \
  --rounds 3 --requests 10 --warmup-waves 5 \
  --output .ci-results/iris/graph-ab-01 \
  2>&1 | tee .ci-results/iris/graph-ab-01.log


## Download available evidence, including failures


In [ ]:
import zipfile
from pathlib import Path

from google.colab import files

root = Path('/content/mstar-iris/.ci-results/iris')
archive = Path('/content/iris-graph-experiment-01.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for name in ('graph-qualify-01', 'graph-audit-01', 'graph-ab-01'):
        directory = root / name
        if directory.exists():
            for path in sorted(directory.rglob('*')):
                if path.is_file() and 'inductor' not in path.parts:
                    bundle.write(path, path.relative_to(root))
        log = root / (name + '.log')
        if log.exists():
            bundle.write(log, log.relative_to(root))
files.download(str(archive))
